# Lab Session 4 — RAG with RDF/SPARQL and a Local Small LLM


## Installation


In [1]:
!pip install rdflib requests


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Configuration


In [10]:
import re
import json
import requests
from typing import List, Tuple
from rdflib import Graph

# ── Configuration ──
NT_FILE      = 'expanded_kb.nt'         
OLLAMA_URL   = 'http://localhost:11434/api/generate'      
MODEL_NAME = "llama3.2:3b"  
MAX_PREDICATES = 80
MAX_CLASSES    = 40
SAMPLE_TRIPLES = 20

print(f'Modèle LLM configuré : {MODEL_NAME}')
print(f'Fichier RDF          : {NT_FILE}')
print(f'URL Ollama           : {OLLAMA_URL}')


Modèle LLM configuré : llama3.2:3b
Fichier RDF          : expanded_kb.nt
URL Ollama           : http://localhost:11434/api/generate


## Vérification d'Ollama


In [11]:
def check_ollama():
    """Vérifie qu'Ollama est accessible."""
    try:
        r = requests.get('http://localhost:11434', timeout=5)
        if 'Ollama' in r.text or r.status_code == 200:
            print(' Ollama est lancé et accessible')
            return True
    except Exception as e:
        print(f' Ollama non accessible : {e}')
        print('  → Lance "ollama serve" dans un terminal')
        return False

check_ollama()


✓ Ollama est lancé et accessible


True


## appel au LLM local (Ollama)


In [20]:
def ask_local_llm(prompt: str, model: str = MODEL_NAME) -> str:
    payload = {
        'model':  model,
        'prompt': prompt,
        'stream': False,
    }
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=600) 
        if response.status_code != 200:
            raise RuntimeError(f'Ollama API error {response.status_code}: {response.text}')
        return response.json().get('response', '')
    except Exception as e:
        return f"[ERREUR] {e}"

In [21]:
print('Test du LLM...')
test_response = ask_local_llm('Réponds en une phrase : qui est Michael Jackson ?')
print(f'Réponse : {test_response[:200]}')


Test du LLM...
Réponse : Michael Jackson était un chanteur, danseur et producteur américain considéré comme l'un des artistes les plus influents de l'histoire de la musique populaire.



## Chargement du graphe RDF

In [22]:
def load_graph(path: str) -> Graph:
    """Charge un fichier RDF (N-Triples ou Turtle) avec rdflib."""
    g = Graph()
    # Détecter le format automatiquement
    fmt = 'nt' if path.endswith('.nt') else 'turtle'
    g.parse(path, format=fmt)
    print(f'✓ Graphe chargé : {len(g)} triplets depuis {path}')
    return g

g = load_graph(NT_FILE)


✓ Graphe chargé : 66719 triplets depuis expanded_kb.nt



## Construction du résumé de schéma


In [23]:
def get_prefix_block(g: Graph) -> str:
    """Collecte les préfixes du graphe et ajoute les préfixes standard."""
    defaults = {
        'rdf':  'http://www.w3.org/1999/02/22-rdf-syntax-ns#',
        'rdfs': 'http://www.w3.org/2000/01/rdf-schema#',
        'xsd':  'http://www.w3.org/2001/XMLSchema#',
        'owl':  'http://www.w3.org/2002/07/owl#',
        'wd':   'http://www.wikidata.org/entity/',
        'wdt':  'http://www.wikidata.org/prop/direct/',
        'mj':   'http://mjackson.org/resource/',
        'prop': 'http://mjackson.org/property/',
    }
    ns_map = {p: str(ns) for p, ns in g.namespace_manager.namespaces()}
    for k, v in defaults.items():
        ns_map.setdefault(k, v)
    return '\n'.join(sorted(f'PREFIX {p}: <{ns}>' for p, ns in ns_map.items()))


def list_distinct_predicates(g: Graph, limit=MAX_PREDICATES) -> List[str]:
    """Liste les prédicats distincts du graphe."""
    q = f'SELECT DISTINCT ?p WHERE {{ ?s ?p ?o . }} LIMIT {limit}'
    return [str(row.p) for row in g.query(q)]


def list_distinct_classes(g: Graph, limit=MAX_CLASSES) -> List[str]:
    """Liste les classes distinctes du graphe"""
    q = f'SELECT DISTINCT ?cls WHERE {{ ?s a ?cls . }} LIMIT {limit}'
    return [str(row.cls) for row in g.query(q)]


def sample_triples(g: Graph, limit=SAMPLE_TRIPLES) -> List[Tuple[str, str, str]]:
    """Retourne un échantillon de triplets du graphe"""
    q = f'SELECT ?s ?p ?o WHERE {{ ?s ?p ?o . }} LIMIT {limit}'
    return [(str(r.s), str(r.p), str(r.o)) for r in g.query(q)]


def build_schema_summary(g: Graph) -> str:
    """Construit le résumé complet du schéma pour le prompt LLM"""
    prefixes    = get_prefix_block(g)
    preds       = list_distinct_predicates(g)
    classes     = list_distinct_classes(g)
    samples     = sample_triples(g)

    pred_lines   = '\n'.join(f'- {p}' for p in preds)
    class_lines  = '\n'.join(f'- {c}' for c in classes)
    sample_lines = '\n'.join(f'- {s} {p} {o}' for s, p, o in samples)

    return f"""
{prefixes}

# Predicates (sampled, unique up to {MAX_PREDICATES})
{pred_lines}

# Classes / rdf:type (sampled, unique up to {MAX_CLASSES})
{class_lines}

# Sample triples (up to {SAMPLE_TRIPLES})
{sample_lines}
""".strip()


schema = build_schema_summary(g)
print('Résumé du schéma construit.')
print(f'Longueur : {len(schema)} caractères')
print('\n--- Extrait ---')
print(schema[:800])


Résumé du schéma construit.
Longueur : 7487 caractères

--- Extrait ---
PREFIX brick: <https://brickschema.org/schema/Brick#>
PREFIX csvw: <http://www.w3.org/ns/csvw#>
PREFIX dc: <http://purl.org/dc/elements/1.1/>
PREFIX dcam: <http://purl.org/dc/dcam/>
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX dcmitype: <http://purl.org/dc/dcmitype/>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX doap: <http://usefulinc.com/ns/doap#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX mj: <http://mjackson.org/resource/>
PREFIX odrl: <http://www.w3.org/ns/odrl/2/>
PREFIX org: <http://www.w3.org/ns/org#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX prof: <http://www.w3.org/ns/dx/prof/>
PREFIX prop: <http://mjackson.org/property/>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX qb: <http://purl.org/linked-data/c


## Baseline : LLM sans KB 

In [18]:
def answer_no_rag(question: str) -> str:
    """Pose une question directement au LLM sans contexte KB."""
    prompt = f'Answer the following question as best as you can:\n\n{question}'
    return ask_local_llm(prompt)


TEST_QUESTIONS = [
    'Which music genre is associated with Michael Jackson (Q2831)?',
    'Which record label did Michael Jackson work with?',
    'Who influenced Michael Jackson according to the knowledge base?',
    'What awards did Michael Jackson receive?',
    'Which artists have the same music style as Michael Jackson?',
]

print('=== BASELINE — Réponses LLM sans KB ===')
baseline_answers = {}
for q in TEST_QUESTIONS:
    print(f'\nQ: {q}')
    ans = answer_no_rag(q)
    baseline_answers[q] = ans
    print(f'A: {ans[:300]}')
    print('-' * 60)


=== BASELINE — Réponses LLM sans KB ===

Q: Which music genre is associated with Michael Jackson (Q2831)?
A: The music genres most commonly associated with Michael Jackson are Pop and R&B. However, his work also incorporated elements of Rock, Soul, Funk, and Dance music.

Michael Jackson was a key figure in the development of the pop-R&B sound of the 1970s and 1980s, and his music often blended these style
------------------------------------------------------------

Q: Which record label did Michael Jackson work with?
A: Michael Jackson worked with several record labels throughout his career, but some of the most notable ones include:

1. Motown Records (1969-1975): This was Jackson's first major record label, where he rose to fame as a member of The Jackson 5.
2. Epic Records (1975-1993): After leaving Motown, Jack
------------------------------------------------------------

Q: Who influenced Michael Jackson according to the knowledge base?
A: Michael Jackson (1958-2009) was a globa


## Génération de SPARQL depuis le langage naturel


In [32]:
SPARQL_INSTRUCTIONS = """
You are a SPARQL generator. Convert the user QUESTION into a valid SPARQL 1.1 SELECT query
for the given RDF graph schema. Follow strictly:
- Use ONLY the IRIs/prefixes visible in the SCHEMA SUMMARY.
- The main entities use prefix wd: (e.g. wd:Q2831 for Michael Jackson).
- Do NOT invent new predicates or classes.
- Return ONLY the SPARQL query in a single fenced code block labeled ```sparql
- No explanations or extra text outside the code block.

IMPORTANT PROPERTIES for Michael Jackson (wd:Q2831):
- music genre    → wdt:P136
- record label   → wdt:P264
- award received → wdt:P166
- date of birth  → wdt:P569
- occupation     → wdt:P106

EXAMPLE:
Q: Which music genre is associated with Michael Jackson?
```sparql
SELECT ?genre WHERE { wd:Q2831 wdt:P136 ?genre . }
```
"""
CODE_BLOCK_RE = re.compile(r"```(?:sparql)?\s*(.*?)```", re.IGNORECASE | re.DOTALL)


def extract_sparql(text: str) -> str:
    m = CODE_BLOCK_RE.search(text)
    if m:
        sparql = m.group(1).strip()
        last_brace = sparql.rfind("}")
        if last_brace != -1:
            sparql = sparql[:last_brace + 1].strip()
        return sparql
    lines = []
    in_sparql = False
    for line in text.strip().splitlines():
        upper = line.upper().strip()
        if upper.startswith("SELECT") or upper.startswith("PREFIX"):
            in_sparql = True
        if in_sparql:
            lines.append(line)
        if in_sparql and line.strip() == "}":
            break  
    return "\n".join(lines).strip() or text.strip()
    
def make_sparql_prompt(schema_summary: str, question: str) -> str:
    """Construit le prompt complet pour la génération SPARQL."""
    return f"""{SPARQL_INSTRUCTIONS}

SCHEMA SUMMARY:
{schema_summary}

QUESTION:
{question}

Return only the SPARQL query in a code block.
"""


def generate_sparql(question: str, schema_summary: str) -> str:
    """Génère une requête SPARQL depuis une question en langage naturel."""
    raw   = ask_local_llm(make_sparql_prompt(schema_summary, question))
    query = extract_sparql(raw)
    return query


print('Test de génération SPARQL...')
test_q = TEST_QUESTIONS[0]
test_sparql = generate_sparql(test_q, schema)
print(f'Question : {test_q}')
print(f'SPARQL généré :\n{test_sparql}')


Test de génération SPARQL...
Question : Which music genre is associated with Michael Jackson (Q2831)?
SPARQL généré :
SELECT ?genre WHERE { wd:Q2831 wdt:P136 ?genre . }



## exécution SPARQL avec rdflib



In [33]:
def run_sparql(g: Graph, query: str) -> Tuple[List[str], List[Tuple]]:
    """Exécute une requête SPARQL avec injection de préfixes et gestion d'erreurs."""
    prefixes = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
"""
    # Nettoyage : on s'assure que la requête commence bien au SELECT
    start_idx = query.upper().find("SELECT")
    clean_body = query[start_idx:] if start_idx != -1 else query
    
    full_query = prefixes + "\n" + clean_body
    
    res = g.query(full_query)
    vars_ = [str(v) for v in res.vars]
    rows = [tuple(str(cell) for cell in r) for r in res]
    return vars_, rows

print('Test d\'exécution SPARQL...')


test_manuel = "SELECT ?p ?o WHERE { wd:Q2831 ?p ?o . }"

try:
    vars_, rows = run_sparql(g, test_manuel)
    print(f'✓ Succès technique ! {len(rows)} résultats trouvés pour MJ.')
    print(f'Variables : {vars_}')
    for row in rows[:3]:
        print(f'  {row}')
        
    print('\nTest de la requête générée par l\'IA...')
    v_ai, r_ai = run_sparql(g, test_sparql)
    print(f'Résultats IA : {len(r_ai)}')

except Exception as e:
    print(f'✗ Erreur : {e}')

Test d'exécution SPARQL...
✓ Succès technique ! 153 résultats trouvés pour MJ.
Variables : ['p', 'o']
  ('http://www.wikidata.org/prop/direct/P737', 'http://www.wikidata.org/entity/Q295919')
  ('http://www.wikidata.org/prop/direct/P737', 'http://www.wikidata.org/entity/Q181483')
  ('http://www.wikidata.org/prop/direct/P737', 'http://www.wikidata.org/entity/Q100937')

Test de la requête générée par l'IA...
Résultats IA : 14


In [34]:
def run_sparql(g: Graph, query: str) -> Tuple[List[str], List[Tuple]]:
    prefixes = """
    PREFIX wd: <http://www.wikidata.org/entity/>
    PREFIX wdt: <http://www.wikidata.org/prop/direct/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    """
    
    query_clean = extract_sparql_from_text(query)
    
    if "PREFIX" not in query_clean.upper():
        final_query = prefixes + "\n" + query_clean
    else:
        final_query = query_clean

    res = g.query(final_query)
    vars_ = [str(v) for v in res.vars]
    rows = [tuple(str(cell) for cell in r) for r in res]
    return vars_, rows

In [35]:
def run_sparql(g: Graph, query: str) -> Tuple[List[str], List[Tuple]]:
    prefixes = """PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
"""
    lines = [l for l in query.splitlines() if not l.strip().upper().startswith("PREFIX")]
    clean_query = "\n".join(lines).strip()
    
    full_query = prefixes + clean_query
    
    res = g.query(full_query)
    vars_ = [str(v) for v in res.vars]
    rows = [tuple(str(cell) for cell in r) for r in res]
    return vars_, rows
    
print('Test d\'exécution SPARQL...')
try:
    vars_, rows = run_sparql(g, test_sparql)
    print(f'✓ Requête exécutée : {len(rows)} résultats')
    print(f'Variables : {vars_}')
    for row in rows[:5]:
        print(f'  {row}')
except Exception as e:
    print(f'✗ Erreur d\'exécution : {e}')
    print('→ La boucle de self-repair sera déclenchée')


Test d'exécution SPARQL...
✓ Requête exécutée : 14 résultats
Variables : ['genre']
  ('http://www.wikidata.org/entity/Q11401',)
  ('http://www.wikidata.org/entity/Q45981',)
  ('http://www.wikidata.org/entity/Q58339',)
  ('http://www.wikidata.org/entity/Q840065',)
  ('http://www.wikidata.org/entity/Q2529400',)


### Self-Repair Loop


In [36]:
REPAIR_INSTRUCTIONS = """
The previous query failed or returned 0 results. 
FIX IT NOW:
- Use wd:Q2831 for Michael Jackson.
- Use wdt:P136 for genre, wdt:P264 for label, wdt:P737 for influence.
- Example for genre: SELECT ?g WHERE { wd:Q2831 wdt:P136 ?g . }
Return ONLY the corrected SPARQL.
"""


def repair_sparql(schema_summary: str, question: str,
                  bad_query: str, error_msg: str) -> str:
    """Demande au LLM de corriger une requête SPARQL invalide."""
    prompt = f"""{REPAIR_INSTRUCTIONS}

SCHEMA SUMMARY:
{schema_summary}

ORIGINAL QUESTION:
{question}

BAD SPARQL:
{bad_query}

ERROR MESSAGE:
{error_msg}

Return only the corrected SPARQL in a code block.
"""
    raw = ask_local_llm(prompt)
    return extract_sparql(raw)



## RAG complet



In [37]:
def answer_with_sparql_generation(g: Graph, schema_summary: str,
                                  question: str,
                                  try_repair: bool = True) -> dict:
    """
    Pipeline RAG complet :
    1. Génère SPARQL depuis la question
    2. Exécute sur le graphe
    3. Self-repair si erreur
    """
    sparql = generate_sparql(question, schema_summary)
    try:
        vars_, rows = run_sparql(g, sparql)
        return {
            'question': question,
            'query':    sparql,
            'vars':     vars_,
            'rows':     rows,
            'repaired': False,
            'error':    None
        }
    except Exception as e:
        err = str(e)
        if try_repair:
            print(f'  Erreur SPARQL, tentative de self-repair...')
            repaired = repair_sparql(schema_summary, question, sparql, err)
            try:
                vars_, rows = run_sparql(g, repaired)
                return {
                    'question': question,
                    'query':    repaired,
                    'vars':     vars_,
                    'rows':     rows,
                    'repaired': True,
                    'error':    None
                }
            except Exception as e2:
                return {
                    'question': question,
                    'query':    repaired,
                    'vars':     [],
                    'rows':     [],
                    'repaired': True,
                    'error':    str(e2)
                }
        return {
            'question': question,
            'query':    sparql,
            'vars':     [],
            'rows':     [],
            'repaired': False,
            'error':    err
        }


def pretty_print_result(result: dict):
    """Affiche les résultats de manière lisible."""
    print(f'\nQuestion : {result["question"]}')
    if result.get('error'):
        print(f'[Erreur] {result["error"]}')
    print(f'[Repaired] {result["repaired"]}')
    print(f'[SPARQL]\n{result["query"]}')
    vars_ = result.get('vars', [])
    rows  = result.get('rows', [])
    if not rows:
        print('[Aucun résultat]')
        return
    print(f'\n[Résultats] ({len(rows)} lignes)')
    print(' | '.join(vars_))
    for r in rows[:20]:
        print(' | '.join(r))
    if len(rows) > 20:
        print(f'... (affichage limité à 20 sur {len(rows)})')



## Évaluation : Baseline vs RAG sur 5 questions



In [38]:
import pandas as pd

rag_results = []
eval_table  = []

for question in TEST_QUESTIONS:
    print(f'\n{'='*60}')
    print(f'Question : {question}')


    print('\n--- Baseline (No RAG) ---')
    baseline = answer_no_rag(question)
    print(baseline[:300])

    print('\n--- SPARQL-generation RAG ---')
    result = answer_with_sparql_generation(g, schema, question, try_repair=True)
    pretty_print_result(result)
    rag_results.append(result)

    n_rows = len(result.get('rows', []))
    eval_table.append({
        'Question':       question[:60] + '...' if len(question) > 60 else question,
        'Baseline':       baseline[:80] + '...' if len(baseline) > 80 else baseline,
        'RAG résultats':  f'{n_rows} ligne(s)' if n_rows > 0 else 'Aucun',
        'Repaired':       result['repaired'],
        'Erreur':         'Oui' if result['error'] else 'Non',
    })

print('\n\n=== TABLEAU D\'ÉVALUATION ===')
df_eval = pd.DataFrame(eval_table)
print(df_eval.to_string(index=False))
df_eval.to_csv('rag_evaluation.csv', index=False)
print('\nrag_evaluation.csv sauvegardé ✓')



Question : Which music genre is associated with Michael Jackson (Q2831)?

--- Baseline (No RAG) ---
Michael Jackson was primarily associated with the pop genre, but his work also incorporated elements of R&B, rock, and dance music. He is often credited with helping to popularize the genres of urban contemporary and adult contemporary music.

--- SPARQL-generation RAG ---

Question : Which music genre is associated with Michael Jackson (Q2831)?
[Repaired] False
[SPARQL]
SELECT ?genre WHERE { wd:Q2831 wdt:P136 ?genre . }

[Résultats] (14 lignes)
genre
http://www.wikidata.org/entity/Q11401
http://www.wikidata.org/entity/Q45981
http://www.wikidata.org/entity/Q58339
http://www.wikidata.org/entity/Q840065
http://www.wikidata.org/entity/Q2529400
http://www.wikidata.org/entity/Q11403
http://www.wikidata.org/entity/Q37073
http://www.wikidata.org/entity/Q9759
http://www.wikidata.org/entity/Q211756
http://www.wikidata.org/entity/Q164444
http://www.wikidata.org/entity/Q131272
http://www.wikidata.


## Demo CLI interactive


In [ ]:
def run_cli_demo(g: Graph, schema_summary: str):
    """Interface CLI interactive pour le chatbot RAG."""
    print('=' * 60)
    print('Chatbot RAG — Michael Jackson Knowledge Base')
    print('Tape "quit" pour quitter')
    print('=' * 60)

    while True:
        try:
            q = input('\nQuestion : ').strip()
        except EOFError:
            break
        if q.lower() in ('quit', 'exit', 'q'):
            print('Au revoir !')
            break
        if not q:
            continue

        print('\n--- Baseline (LLM seul) ---')
        print(answer_no_rag(q))

        print('\n--- RAG (SPARQL + KB) ---')
        result = answer_with_sparql_generation(g, schema_summary, q, try_repair=True)
        pretty_print_result(result)

run_cli_demo(g, schema)



Chatbot RAG — Michael Jackson Knowledge Base
Tape "quit" pour quitter



Question :  when born Mickael Jackson?



--- Baseline (LLM seul) ---
Michael Jackson was born on August 29, 1958.

--- RAG (SPARQL + KB) ---

Question : when born Mickael Jackson?
[Repaired] False
[SPARQL]
SELECT ?date WHERE { wd:Q2831 wdt:P569 ?date . }
[Aucun résultat]



Question :  What awards did Michael Jackson receive?



--- Baseline (LLM seul) ---
Michael Jackson was a highly acclaimed and award-winning artist. Here are some of his notable awards:

1. Grammy Awards: He won 13 Grammy Awards out of 38 nominations, including Album of the Year for "Thriller" (1984), Record of the Year for "Billie Jean" (1984), and Best Music Video, Short Form for "Bad" (1988).
2. American Music Awards: He won 16 American Music Awards, including Artist of the Century (2006) and Favorite Pop/Rock Male Artist (1987, 1991, 1993, 1995, 1999).
3. Billboard Music Awards: He was awarded the Billboard Millennium Award (2000), Billboard Touring Artist of the Decade (2002), and was named the Billboard Top Artist of All-Time (2008).
4. MTV Video Music Awards: He won 15 MTV Video Music Awards, including Best Male Video for "Black or White" (1991) and Best Choreography in a Video for "Thriller" (1985).
5. Guinness World Records: He was awarded several Guinness World Records, including Most Successful Entertainer of All Time (2002), Hi

### Exemples de démonstration 

In [41]:
#question sur les genres musicaux
demo_q1 = 'What music genres are associated with Michael Jackson (Q2831)?'
print(f'=== EXEMPLE 1 ===')
print(f'Question : {demo_q1}')
print('\n--- Baseline ---')
print(answer_no_rag(demo_q1)[:400])
print('\n--- RAG ---')
r1 = answer_with_sparql_generation(g, schema, demo_q1)
pretty_print_result(r1)


=== EXEMPLE 1 ===
Question : What music genres are associated with Michael Jackson (Q2831)?

--- Baseline ---
Michael Jackson's music spans multiple genres, but he is primarily associated with the following styles:

1. Pop: Michael Jackson is widely regarded as one of the most successful pop artists of all time, with hits like "Billie Jean," "Beat It," and "Thriller."
2. R&B (Rhythm and Blues): As a member of The Jackson 5, Michael Jackson's music was heavily influenced by soul and R&B, with songs like "I

--- RAG ---

Question : What music genres are associated with Michael Jackson (Q2831)?
[Repaired] False
[SPARQL]
SELECT ?genre WHERE { wd:Q2831 wdt:P136 ?genre . }

[Résultats] (14 lignes)
genre
http://www.wikidata.org/entity/Q11401
http://www.wikidata.org/entity/Q45981
http://www.wikidata.org/entity/Q58339
http://www.wikidata.org/entity/Q840065
http://www.wikidata.org/entity/Q2529400
http://www.wikidata.org/entity/Q11403
http://www.wikidata.org/entity/Q37073
http://www.wikidata.org/

In [42]:
#question sur les awards
demo_q2 = 'Which awards did Michael Jackson (Q2831) receive according to the knowledge base?'
print(f'=== EXEMPLE 2 ===')
print(f'Question : {demo_q2}')
print('\n--- Baseline ---')
print(answer_no_rag(demo_q2)[:400])
print('\n--- RAG ---')
r2 = answer_with_sparql_generation(g, schema, demo_q2)
pretty_print_result(r2)


=== EXEMPLE 2 ===
Question : Which awards did Michael Jackson (Q2831) receive according to the knowledge base?

--- Baseline ---
I'm not able to access the specific information about Michael Jackson's awards in my current knowledge cutoff. However, I can suggest some possible sources where you may be able to find this information.

Michael Jackson was a renowned musician and artist who received numerous awards and accolades throughout his career. He won 13 Grammy Awards, among many others. If you're looking for more specifi

--- RAG ---

Question : Which awards did Michael Jackson (Q2831) receive according to the knowledge base?
[Repaired] False
[SPARQL]
SELECT ?award WHERE { wd:Q2831 wdt:P166 ?award . }

[Résultats] (22 lignes)
award
http://www.wikidata.org/entity/Q1459443
http://www.wikidata.org/entity/Q953746
http://www.wikidata.org/entity/Q1027904
http://www.wikidata.org/entity/Q19858142
http://www.wikidata.org/entity/Q135498
http://www.wikidata.org/entity/Q1333798
http://www.wikid

## Script Python standalone 

In [45]:
script = '''
import re
import requests
from typing import List, Tuple
from rdflib import Graph

NT_FILE      = 'expanded_kb.nt'
OLLAMA_URL   = 'http://localhost:11434/api/generate'
MODEL_NAME   = 'gemma:2b'
MAX_PREDICATES = 80
MAX_CLASSES    = 40
SAMPLE_TRIPLES = 20

def ask_local_llm(prompt, model=MODEL_NAME):
    payload = {'model': model, 'prompt': prompt, 'stream': False}
    r = requests.post(OLLAMA_URL, json=payload, timeout=120)
    return r.json().get('response', '') if r.status_code == 200 else ''

def load_graph(path):
    g = Graph()
    fmt = 'nt' if path.endswith('.nt') else 'turtle'
    g.parse(path, format=fmt)
    print(f'Loaded {len(g)} triples from {path}')
    return g

def build_schema_summary(g):
    preds = [str(r.p) for r in g.query(f'SELECT DISTINCT ?p WHERE {{ ?s ?p ?o }} LIMIT {MAX_PREDICATES}')]
    clss  = [str(r.cls) for r in g.query(f'SELECT DISTINCT ?cls WHERE {{ ?s a ?cls }} LIMIT {MAX_CLASSES}')]
    samps = [(str(r.s), str(r.p), str(r.o)) for r in g.query(f'SELECT ?s ?p ?o WHERE {{ ?s ?p ?o }} LIMIT {SAMPLE_TRIPLES}')]
    ns    = {p: str(ns) for p, ns in g.namespace_manager.namespaces()}
    for k, v in [('wd','http://www.wikidata.org/entity/'),('wdt','http://www.wikidata.org/prop/direct/')]:
        ns.setdefault(k, v)
    prefixes     = chr(10).join(sorted(f'PREFIX {p}: <{n}>' for p, n in ns.items()))
    pred_lines   = chr(10).join(f'- {p}' for p in preds)
    class_lines  = chr(10).join(f'- {c}' for c in clss)
    sample_lines = chr(10).join(f'- {s} {p} {o}' for s,p,o in samps)
    return f'{prefixes}\\n\\n# Predicates\\n{pred_lines}\\n\\n# Classes\\n{class_lines}\\n\\n# Samples\\n{sample_lines}'

CODE_BLOCK_RE = re.compile(r'```(?:sparql)?\\s*(.*?)```', re.IGNORECASE | re.DOTALL)

def extract_sparql(text):
    m = CODE_BLOCK_RE.search(text)
    if m:
        sparql = m.group(1).strip()
        last_brace = sparql.rfind("}")
        if last_brace != -1:
            sparql = sparql[:last_brace + 1].strip()
        return sparql
    lines = []
    in_sparql = False
    for line in text.strip().splitlines():
        upper = line.upper().strip()
        if upper.startswith("SELECT") or upper.startswith("PREFIX"):
            in_sparql = True
        if in_sparql:
            lines.append(line)
        if in_sparql and line.strip() == "}":
            break
    return "\\n".join(lines).strip() or text.strip()

SPARQL_INSTRUCTIONS = """You are a SPARQL generator. Convert the QUESTION into a valid SPARQL 1.1 SELECT query.
IMPORTANT PROPERTIES for Michael Jackson (wd:Q2831):
- music genre    -> wdt:P136
- record label   -> wdt:P264
- award received -> wdt:P166
- date of birth  -> wdt:P569
- date of death  -> wdt:P570
- occupation     -> wdt:P106
- influenced by  -> wdt:P737
EXAMPLE:
Q: What is the music genre of Michael Jackson?
SELECT ?genre WHERE { wd:Q2831 wdt:P136 ?genre . }
Return only a sparql code block, nothing after the closing backticks."""

def generate_sparql(question, schema):
    raw = ask_local_llm(f'{SPARQL_INSTRUCTIONS}\\n\\nSCHEMA:\\n{schema}\\n\\nQUESTION:\\n{question}')
    return extract_sparql(raw)

def run_sparql(g, query):
    prefixes = """PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
"""
    lines = [l for l in query.splitlines() if not l.strip().upper().startswith("PREFIX")]
    full_query = prefixes + "\\n".join(lines).strip()
    res = g.query(full_query)
    return [str(v) for v in res.vars], [tuple(str(c) for c in r) for r in res]

def repair_sparql(schema, question, bad_query, error):
    prompt = f'Fix this SPARQL query.\\nSCHEMA:\\n{schema}\\nQUESTION:\\n{question}\\nBAD SPARQL:\\n{bad_query}\\nERROR:\\n{error}\\nReturn only corrected SPARQL in a code block.'
    return extract_sparql(ask_local_llm(prompt))

def rag_answer(g, schema, question):
    sparql = generate_sparql(question, schema)
    try:
        return run_sparql(g, sparql), sparql, False
    except Exception as e:
        fixed = repair_sparql(schema, question, sparql, str(e))
        try:
            return run_sparql(g, fixed), fixed, True
        except:
            return ([], []), fixed, True

if __name__ == '__main__':
    g = load_graph(NT_FILE)
    schema = build_schema_summary(g)
    print('Michael Jackson KB Chatbot — type quit to exit')
    while True:
        q = input('\\nQuestion: ').strip()
        if q.lower() == 'quit': break
        print('\\n[Baseline]', ask_local_llm(f'Answer: {q}')[:200])
        (vars_, rows), sparql, repaired = rag_answer(g, schema, q)
        print(f'\\n[RAG] (repaired={repaired})')
        print(sparql)
        if rows:
            print(' | '.join(vars_))
            for r in rows[:10]: print(' | '.join(r))
        else:
            print('No results.')
'''

with open('lab_rag_sparql_gen.py', 'w', encoding='utf-8') as f:
    f.write(script)
print('lab_rag_sparql_gen.py sauvegardé ✓')

lab_rag_sparql_gen.py sauvegardé ✓


In [44]:
readme = '''# RAG with RDF/SPARQL and Local LLM

## Requirements
- Python >= 3.9
- `pip install rdflib requests pandas`
- [Ollama](https://ollama.com) installed and running

## Setup
1. Start Ollama service: `ollama serve`
2. Download a model: `ollama pull gemma:2b`
3. Place `expanded_kb.nt` in the same folder

## Run
```bash
python lab_rag_sparql_gen.py
```

## Model configuration
Edit `MODEL_NAME` in the script to change the LLM:
- `gemma:2b` (default)
- `deepseek-r1:1.5b`
- `llama3.2:1b`

## Architecture
1. Load RDF graph (expanded_kb.nt)
2. Build schema summary (prefixes, predicates, classes, sample triples)
3. User asks a question in natural language
4. LLM generates SPARQL from question + schema
5. Execute SPARQL on local graph with rdflib
6. If SPARQL fails → self-repair loop
7. Return grounded results
'''

with open('README.md', 'w', encoding='utf-8') as f:
    f.write(readme)
print('README.md sauvegardé ✓')


README.md sauvegardé ✓
